In [9]:
import numpy as np 
import matplotlib.pyplot as plt
from tdmclient import ClientAsync, aw
import math
from VisibilityGraph import *
from kalman_filter import *
from Vision import *

In [ ]:
client = ClientAsync()
node = aw(client.wait_for_node())
await node.lock()

In [4]:
global_path = [(1, 10), (5, 6), (10, 10)]
thymio_pose = (3, 2)
v = 100

In [5]:
def closest_point_on_path(path, pose):
    """
    Find the closest point on a polyline path to a given pose.

    path: list of (x, y) waypoints
    pose: (x, y) or (x, y, theta); only x and y are used

    Returns: (closest_point, distance, segment_index)
    """
    px, py = pose[0], pose[1]

    if len(path) == 1:
        return path[0], math.hypot(px - path[0][0], py - path[0][1]), 0

    best_point, best_dist, best_idx = None, float("inf"), 0

    for i in range(len(path) - 1):
        ax, ay = path[i]
        bx, by = path[i + 1]
        abx, aby = bx - ax, by - ay
        seg_len_sq = abx**2 + aby**2

        if seg_len_sq == 0:          # degenerate segment (duplicate point)
            t = 0.0
        else:
            # Projection parameter of the pose onto the segment, clamped to [0, 1]
            t = ((px - ax) * abx + (py - ay) * aby) / seg_len_sq
            t = max(0.0, min(1.0, t))

        cx, cy = ax + t * abx, ay + t * aby
        d = math.hypot(px - cx, py - cy)

        if d < best_dist:
            best_point, best_dist, best_idx = (cx, cy), d, i

    return best_point, best_dist, best_idx

In [7]:
def wrap_angle(angle):
    """Wrap angle to [-pi, pi]."""
    return math.atan2(math.sin(angle), math.cos(angle))

def compute_heading(a, b, prev_heading):
    dx = b[0] - a[0]
    dy = b[1] - a[1]

    distance = math.hypot(dx, dy)
    abs_heading = math.atan2(dy, dx)          # absolute heading of this segment
    turn = wrap_angle(abs_heading - prev_heading)  # signed turn needed from current heading

    return distance, turn, abs_heading

In [8]:
SPEED_TO_MM_S = 0.335   
WHEEL_BASE = 95.0

async def set_speed(left, right):
    await node.set_variables({
        "motor.left.target": [int(left)],
        "motor.right.target": [int(right)]
    })
    
async def forward(v, d): # [mm/s, mm]
    T = d/v
    thymio_v = v/SPEED_TO_MM_S
    
    try:
        await set_speed(thymio_v, thymio_v)
        await client.sleep(T)          
    finally:
        await set_speed(0, 0)
        
async def turn(v, theta): # [rad/s, rad]

    arc = abs(theta) * WHEEL_BASE / 2 
    T = arc / v                          
    thymio_v =  abs(v) / SPEED_TO_MM_S         
    direction = 1 if theta > 0 else -1
    try:
        await set_speed(-direction * thymio_v, direction * thymio_v)
        await client.sleep(T)
    finally:
        await set_speed(0, 0)

In [6]:
point, dist, seg = closest_point_on_path(global_path, thymio_pose)
print(point, dist, seg)  
prev_heading = 0

distance, heading, prev_heading = compute_heading(thymio_pose, point, prev_heading)
await turn(v, heading)
await forward(v, distance)

(5.0, 6.0) 4.47213595499958 0
